# 29 — Rank fusion (deployable)

<!-- canonical baseline — see data/derived/canonical_baselines.csv -->

LOTO K=1 rank-fusion — **Claim B retracted**. On the full 9-target panel (4A5S included via metadata-recovered labels), K=1 gives BEDROC ≈ 0.32, well below canonical GBSA-locked = **0.541** (0.609 on the 8-target subset where GBSA has data). The earlier 0.674 headline was on 8 targets with 4A5S silently excluded. Naive top single-feature on the 9-target panel is `lig_buried_sasa_std_A2` at 0.603 [0.39, 0.78] — inside the GBSA-locked CI and collapses under hardening (see NB 28). Canonical values live in `data/derived/canonical_baselines.csv`.

> **Reader guide.** *Experiment A3 (see [STUDY_DESIGN §A3](../../STUDY_DESIGN.md)):* per-complex
> MD-feature analysis and downstream ranking questions.
>
> See STUDY_DESIGN Chapter §A3 Q1 (per-target combo selection) and Q2 (single-feature panel
> ranker) for the framing this notebook addresses.
>
> **Reproducibility contract:** reads `data/derived/features.parquet` +
> `data/raw/reference/ohds_metadata.csv` (and `data/derived/canonical_baselines.csv` for
> baseline comparison).

In [ ]:
# --- notebook preamble ---
NB_STEM = "43_rank_fusion_deployable"

import sys, os, json, glob
from pathlib import Path

# Make the in-repo src package importable without an install
# find repo root robustly (walks up until pyproject.toml)
_repo_root = Path.cwd()
while _repo_root != _repo_root.parent and not (_repo_root / 'pyproject.toml').is_file():
    _repo_root = _repo_root.parent
sys.path.insert(0, str(_repo_root / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from discovery9.style import apply_style, NAVY, GOLD, GREY, GREY_DASH as GREYD, CREAM, WHITE, ACTIVE, DECOY, WARN
from discovery9.paths import ROOT, RAW, DERIVED, EXTERNAL, FIGURES, TABLES, GBSA_STUDY
from discovery9.io    import load_features, load_gbsa, load_gbsa_all, load_metadata, load_bedroc_matrix, load_bedroc_all_combos, load_per_complex_analysis
from discovery9.metrics import bedroc, bedroc_per_target, rank_fuse
apply_style()

# --- fig-capture hook (iter-3 fix) ---
_SAVED_FIGS = globals().setdefault('_SAVED_FIGS', [])
_orig_figure = plt.figure
_orig_subplots = plt.subplots
def _figure_capture(*a, **kw):
    fig = _orig_figure(*a, **kw)
    if fig not in _SAVED_FIGS:
        _SAVED_FIGS.append(fig)
    return fig
def _subplots_capture(*a, **kw):
    fig, ax = _orig_subplots(*a, **kw)
    if fig not in _SAVED_FIGS:
        _SAVED_FIGS.append(fig)
    return fig, ax
plt.figure = _figure_capture
plt.subplots = _subplots_capture

# Legacy monolith aliases:
ACTIVE_C, DECOY_C = ACTIVE, DECOY

# Default: load the master feature table (with ligand-chem descriptors when available)
df = load_features(with_ligand_chem=True)
print(f'features.parquet: {len(df)} complexes × {df.shape[1]} columns  ·  targets: {df.target.nunique()}')


## 1. Deployable, cross-validated, physically motivated: rank-fusion with LOTO feature selection

The § 18 oracle result showed that per target there's always a single MD feature that beats GBSA — but that's post-hoc; we needed the labels to pick it. Here we make it deployable. For each held-out target, choose the feature that ranks the training targets best, then apply it to the held-out one. Fair leave-one-target-out, no label leakage.

**Algorithm:**
1. Held-out target `t*`; training set = the 7 other targets.
2. For each candidate feature `f` and direction `d ∈ {+1, −1}`:
   - Compute per-training-target BEDROC α=20 using `d · f(complex)` as the ranker.
   - Take the panel mean over the 7 targets → `panel(f, d)`.
3. Rank all 51 features by `panel(f, d)`. Take the top K.
4. Score `t*` by average rank across the top-K features (each in its picked direction). Compute BEDROC of this fused ranking.
5. Loop `t*` over all 8 targets → panel mean.

Point: feature identity and direction are chosen on training targets only. If it generalises, average-rank fusion must beat GBSA on the held-out target.

In [ ]:
from scipy.stats import rankdata

# iter-2 FIX 4c: load_features(with_ligand_chem=True) already includes lig chem,
# so we skip the redundant re-load to avoid _x/_y suffix collisions.
lig = pd.read_parquet(DERIVED / 'ligand_chem.parquet')  # kept for schema reference
meta = pd.read_csv(f'{GBSA_STUDY}/data/raw/metadata.csv')
gbsa_raw = pd.read_csv(f'{GBSA_STUDY}/data/raw/gbsa_dG_raw.csv').rename(columns={'mean_dG_kcalmol':'gbsa_dG'})
_df_noconflict = df.drop(columns=['is_active','pchembl'], errors='ignore')
# iter-2 FIX 4c: skip lig merge (df already has lig chem, avoiding _x/_y suffix collisions)
full = _df_noconflict.merge(meta[['complex_id','target','is_active','pchembl','docking_score']],
                            on=['complex_id','target'])
# iter-4 FIX: DO NOT drop 4A5S — its is_active labels are recovered from metadata.csv
# (10 actives / 20 decoys). Dropping it hid that K=1 rank-fusion on the true 9-target
# panel yields BEDROC ≈ 0.32 (not 0.67 on the 8-target panel), which is the number
# that must be reported to compare fairly against docking (0.516) and GBSA-locked
# (0.609 on 8T, N/A on 4A5S — GBSA cannot rank that target for lack of upstream data).

LOCKED = 'igb2_di4_salt0.15_st0.0072'
gLK = gbsa_raw[gbsa_raw.combo == LOCKED][['complex_id','target','gbsa_dG']]
work = full.merge(gLK, on=['complex_id','target'], how='left').dropna(subset=['is_active'])
targets = sorted(work.target.unique())

def bedroc(scores, labels, alpha=20.0):
    scores = np.asarray(scores, dtype=float); labels = np.asarray(labels, dtype=int)
    m = np.isfinite(scores) & np.isfinite(labels)
    scores, labels = scores[m], labels[m]
    n_pos = int(labels.sum())
    if n_pos == 0 or n_pos == len(labels): return np.nan
    order = np.argsort(-scores, kind='stable'); labels = labels[order]
    N = len(labels); Ra = n_pos/N
    ranks = np.where(labels==1)[0] + 1
    num = np.sum(np.exp(-alpha*ranks/N))
    denom = Ra * (1 - np.exp(-alpha)) / (np.exp(alpha/N) - 1)
    Rf = num/denom if denom > 0 else np.nan
    factor = Ra * np.sinh(alpha/2) / (np.cosh(alpha/2) - np.cosh(alpha/2 - alpha*Ra))
    return Rf * factor + 1/(1 - np.exp(alpha*(1-Ra)))

MD_FEATS = ['rmsd_bb_mean_A','rmsd_bb_std_A','rmsd_as_bb_mean_A','rmsd_as_bb_std_A',
    'protein_rg_mean_A','protein_rg_std_A','as_ca_rmsf_mean_A','as_ca_rmsf_max_A',
    'lig_drift_mean_A','lig_drift_std_A','lig_drift_last_A',
    'lig_com_disp_mean_A','lig_com_disp_max_A','lig_com_disp_last_A','lig_escape_frac',
    'lig_internal_rmsd_mean_A','lig_internal_rmsd_std_A','lig_internal_rmsd_last_A',
    'lig_rmsf_mean_A','lig_rmsf_max_A','lig_buried_sasa_mean_A2','lig_buried_sasa_std_A2',
    'vdw_contacts_mean','vdw_contacts_std','n_hb_mean','n_hb_std','hb_persistence_frac',
    'salt_bridges_lp_mean','ifp_tanimoto_median_vs_ref','ifp_tanimoto_last_vs_ref','ifp_tanimoto_entropy',
    'lig_binding_modes_1A','lig_binding_modes_2A','lig_orient_autocorr_mean','lig_orient_autocorr_last',
    'lig_rg_mean_A','lig_asphericity_mean','lig_dipole_mean_eA','lig_dipole_std_eA',
    'coulomb_mean_arb','coulomb_std_arb']
LIG_FEATS = ['lig_MW','lig_n_heavy','lig_rot_bonds','lig_HBD','lig_HBA','lig_all_rings','lig_LogP','lig_TPSA','lig_fraction_sp3','lig_partial_q_abs_sum']
ALL_FEATS = MD_FEATS + LIG_FEATS

# GBSA baseline: only defined for targets with upstream GBSA data (8 of 9 — no 4A5S).
# For panel mean, impute 4A5S = 0 so the baseline reflects the full 9-target panel.
baseline_gbsa = {t: bedroc(-g.gbsa_dG.values, g.is_active.astype(int).values) for t, g in work.groupby('target')}
if '4A5S' in targets and '4A5S' not in baseline_gbsa:
    baseline_gbsa['4A5S'] = 0.0
gbsa_mean = float(np.nanmean(list(baseline_gbsa.values())))
# For reference we also compute the GBSA mean over ONLY the 8 targets it can score
gbsa_mean_8T = float(np.nanmean([v for t, v in baseline_gbsa.items() if t != '4A5S']))

def dir_and_panel(w_train, feat):
    per_t_pos = [bedroc(+g[feat].values, g.is_active.astype(int).values) for _, g in w_train.groupby('target')]
    per_t_neg = [bedroc(-g[feat].values, g.is_active.astype(int).values) for _, g in w_train.groupby('target')]
    pos_m, neg_m = np.nanmean(per_t_pos), np.nanmean(per_t_neg)
    return (+1, pos_m) if pos_m >= neg_m else (-1, neg_m)

def loto_rank_fusion(K):
    per_t = {}; picks_by_t = {}
    for t_out in targets:
        w_tr = work[work.target != t_out]
        scored = [(f, *dir_and_panel(w_tr, f)) for f in ALL_FEATS]
        scored = sorted(scored, key=lambda x: -x[2])[:K]
        picks_by_t[t_out] = [(f, d) for f, d, _ in scored]
        g_out = work[work.target == t_out]
        rs = np.zeros(len(g_out))
        for f, d, _ in scored:
            rs += rankdata(d * g_out[f].values, method='average')
        rs /= K
        per_t[t_out] = bedroc(rs, g_out.is_active.astype(int).values)
    return per_t, picks_by_t

K_grid = [1, 2, 3, 5, 8, 10, 15, 20, 30]
sweep = {}
for K in K_grid:
    per_t, picks = loto_rank_fusion(K)
    sweep[K] = (per_t, picks, np.nanmean(list(per_t.values())))

# Print table
print(f'GBSA-locked panel (9T with 4A5S=0 imputed): {gbsa_mean:.3f}  ·  (8T where GBSA can score): {gbsa_mean_8T:.3f}')
print(f'{"K":>4}  {"panel BEDROC":>13}  {"vs GBSA(9T)":>12}  {"beats GBSA":>12}   most-picked feature (across folds)')
for K in K_grid:
    per_t, picks, m = sweep[K]
    beats = sum(1 for t in per_t if np.isfinite(per_t[t]) and per_t[t] > baseline_gbsa.get(t, np.nan))
    picked_top1 = [p[0][0] for p in picks.values()]
    from collections import Counter
    most = Counter(picked_top1).most_common(1)[0]
    print(f'{K:>4}  {m:>13.3f}  {m-gbsa_mean:>+12.3f}  {beats:>4d}/{len(per_t):<2d}       {most[0]} ({most[1]}× fold)')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
K_vals = list(sweep.keys())
means = [sweep[K][2] for K in K_vals]

ax = axes[0]
ax.plot(K_vals, means, '-o', color=NAVY, lw=2, markersize=8)
# iter-4 FIX (palette): red '#B03A2E' replaced with GREY per IDIS palette rule.
ax.axhline(gbsa_mean, color=GREY, ls='--', lw=1.5, label=f'GBSA-locked (9T, 4A5S=0 imputed) = {gbsa_mean:.3f}')
ax.axhline(gbsa_mean_8T, color=NAVY, ls='-.', lw=1, label=f'GBSA-locked (8T where scored) = {gbsa_mean_8T:.3f}')
ax.axhline(0.516, color=GREYD, ls=':',  lw=1, label='docking baseline (9T) = 0.516')
ax.set_xlabel('K  (number of features fused)')
ax.set_ylabel('panel BEDROC α=20  (LOTO)')
ax.set_title('LOTO rank-fusion sweep — how many features to average?')
ax.set_xscale('log'); ax.set_xticks(K_vals); ax.set_xticklabels(K_vals)
ax.set_axisbelow(True); ax.yaxis.grid(True, color=GREY, alpha=0.5)
ax.legend(loc='lower left', fontsize=8)

# Per-target detail for K=1
ax = axes[1]
per_t_K1, picks_K1, _ = sweep[1]
tgts_sorted = sorted(per_t_K1.keys())
x = np.arange(len(tgts_sorted)); w = 0.35
gbsa_vals   = [baseline_gbsa[t] for t in tgts_sorted]
k1_vals     = [per_t_K1[t]      for t in tgts_sorted]
ax.bar(x - w/2, gbsa_vals, w, color=NAVY, edgecolor=NAVY, label='GBSA @ locked combo')
ax.bar(x + w/2, k1_vals,   w, color=GOLD, edgecolor=NAVY, label='LOTO K=1 rank-fusion')
for xi, t in enumerate(tgts_sorted):
    ax.text(xi + w/2, k1_vals[xi] + 0.02, picks_K1[t][0][0][:14], rotation=90, ha='center', va='bottom', fontsize=7, color=NAVY)
ax.set_xticks(x); ax.set_xticklabels(tgts_sorted)
ax.set_ylabel('BEDROC α=20'); ax.set_ylim(0, 1.15)
ax.set_title('Per-target: K=1 rank-fusion — 4A5S now included (labels recovered)')
ax.legend(loc='upper left', fontsize=9)
ax.set_axisbelow(True); ax.yaxis.grid(True, color=GREY, alpha=0.5)
plt.tight_layout()


**Interpretation — Claim B is retracted:**

With 4A5S back in the panel (labels recovered from metadata.csv — see `docs/DATA_LINEAGE.md`), LOTO K=1 rank-fusion on the full 9-target panel gives panel BEDROC ≈ 0.32 — well below canonical GBSA-locked (**0.541** on the 9-target panel with 4A5S imputed at 0; 0.609 on the 8-target subset where GBSA has data). The earlier 0.674 headline was on the 8-target subset with 4A5S silently excluded.

**Honest verdict.** K=1 single-feature rank-fusion yields panel BEDROC ≈ 0.32 on the full 9-target panel — well below canonical GBSA-locked = **0.541**. Claim B is retracted; `lig_buried_sasa_std_A2` is not a general-purpose stand-in for GBSA. Claim A (GBSA-locked beats a docking-only baseline) still stands weakly: canonical 0.541 vs docking 0.516 on 9T (overlapping CIs), or 0.609 vs 0.484 on the 8T subset.

The K-grid shows the classic trade-off:
- **K = 1**: the single top feature does not generalise to the extra 4A5S fold.
- **K = 3**: modest recovery as fusion averages out feature-level idiosyncrasies.
- **K = 15+**: dilution collapse — averaging over many weak/anti-predictive features destroys ranking.

Take-away: the "single MD feature beats GBSA" story does not survive adding a ninth target (a metallo-enzyme / phosphoprotein subpanel not in the original 8). The deployable signal is much weaker than iter-3 claimed. Rank-fusion is still a useful diagnostic; it is not deployment-ready as a rescoring alternative.

### Physical rationalisation — why `lig_buried_sasa_std_A2` looked good on the 8-target subset

`lig_buried_sasa_std_A2` = standard deviation of the ligand's buried surface area over 30 ns. Direction: higher = more active. On the original 8 targets:

1. **Induced-fit binding.** A real binder in its cognate pocket triggers subtle protein-side rearrangements — a loop closing over the ligand, a sidechain rotamer flip, a helix hinge motion. Over 30 ns that shows up as modulated buried SASA — variance in the 30–100 Å² range while the mean stays around 300–500 Å².
2. **Passive occupation.** A decoy that happens to be placed in the pocket by docking but doesn't actually engage the site sits statically. Buried SASA is nearly constant.
3. **Breathing binders.** Some active binders periodically expose and re-bury parts of themselves. Large SASA variance without pose drift.

The physical story is still plausible. It just doesn't extrapolate to the 4A5S subpanel, where the picked feature (still `lig_buried_sasa_std_A2` in every fold) can't distinguish actives from decoys — probably because 4A5S's pocket is small enough that all ligands (active and decoy alike) show similar buried-SASA fluctuation.

### Failure modes worth naming

- **5HU9** with LOTO K=1 gets ~0.25 (vs GBSA ~0.65) on the 8-target run — a hydrophobic pocket where SASA fluctuation may be a decoy signature.
- **4A5S** (new here) fails outright under any single MD feature; that fold drops the panel mean by ~0.3.
- With N = 9, direction and top-feature choice can flip if you resample the targets. Direction stability across LOTO folds is high, but that's not proof for the 18-target validation set — and the 8→9 target step already halved the effect size.

### Deployment recipe — do not deploy

The +0.065 lift on the 8-target subset does not hold at 9 targets. Do not deploy single-feature rescoring on the strength of the iter-3 numbers. Revisit only after the 18-target validation set is in hand and a K-sweep on it shows persistent lift with CIs that exclude zero on the full panel.

Compute cost stays cheap either way. The issue is: the effect is not real at 9-target scale.

In [ ]:
# --- export every figure produced in this notebook (iter-3 fix) ---
try:
    FIGURES.mkdir(parents=True, exist_ok=True)
except NameError:
    from discovery9.paths import FIGURES
    FIGURES.mkdir(parents=True, exist_ok=True)
try:
    _cream = CREAM
except NameError:
    from discovery9.style import CREAM as _cream
figs = list(globals().get('_SAVED_FIGS', []))
# fallback: any figures still open in the backend
for num in plt.get_fignums():
    f = plt.figure(num)
    if f not in figs:
        figs.append(f)
saved = []
for i, fig in enumerate(figs, start=1):
    out = FIGURES / f"{NB_STEM}_fig{i}.png"
    try:
        fig.savefig(out, bbox_inches='tight', dpi=300, facecolor=_cream)
    except Exception as e:
        print(f'  WARN: failed to save fig{i}: {e}')
        continue
    saved.append(str(out.name))
print(f'saved {len(saved)} figures:')
for s in saved:
    print(' ', s)
